# 📉 SaaS Churn Prediction — Classification for Revenue Protection

**Session 4 | Industrial Use Case 2 | DSA & ML for Business**

---

### Business Context
- Azure annual revenue: **$80B+** — even **1% churn = $800M** at risk
- Acquiring new enterprise customer costs **5-7x** retaining existing one
- Early warning: **60-90 days** before churn, usage patterns change
- SaaS benchmark: 5-7% annual churn for enterprise, 20-30% for SMB

### What You'll Learn
1. **EDA** — examine churn rate by plan type, strongest predictors
2. **Feature engineering** — create trend features
3. Train **Decision Tree, Random Forest, SGD Classifier**
4. **Precision vs Recall tradeoff** — which metric matters more for churn?
5. **Revenue impact simulation** — if model catches 80% of churners early

## Step 1: Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve,
                             roc_auc_score, precision_recall_curve)
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv("../datasets/saas_churn_prediction.csv")
print(f"Dataset: {df.shape[0]} customers")
print(f"Churn rate: {df['churned'].mean():.2%}")
print(f"\nChurn by plan type:")
print(df.groupby('plan_type')['churned'].agg(['mean', 'count']).round(3))
df.head()

## Step 2: EDA — Churn Predictors & Feature Engineering

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
features = ['login_frequency_30d', 'support_tickets_90d', 'usage_decline_pct', 'days_to_renewal', 'nps_score']
for ax, feat in zip(axes.flat[:5], features):
    for lbl, color, name in [(0, '#10B981', 'Active'), (1, '#EF4444', 'Churned')]:
        ax.hist(df[df['churned']==lbl][feat], bins=30, alpha=0.5, label=name, color=color, density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold'); ax.legend(fontsize=8)

churn_by_plan = df.groupby('plan_type')['churned'].mean()
axes[1, 2].bar(churn_by_plan.index, churn_by_plan.values, color=['#2563EB', '#F59E0B', '#EF4444'])
axes[1, 2].set_title('Churn Rate by Plan Type', fontweight='bold')
plt.suptitle('Churn Pattern Analysis', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

le = LabelEncoder()
df['plan_encoded'] = le.fit_transform(df['plan_type'])
df['login_per_ticket'] = df['login_frequency_30d'] / (df['support_tickets_90d'] + 1)
df['nps_usage_interaction'] = df['nps_score'] * (1 - df['usage_decline_pct'] / 100)

## Step 3: Train Models

In [ ]:
feature_cols = ['login_frequency_30d', 'support_tickets_90d', 'usage_decline_pct',
                'plan_encoded', 'days_to_renewal', 'nps_score',
                'login_per_ticket', 'nps_usage_interaction']
X = df[feature_cols]; y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train); X_test_sc = scaler.transform(X_test)

models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'SGD Classifier': SGDClassifier(loss='log_loss', class_weight='balanced', random_state=42, max_iter=1000),
}
results = {}
for name, model in models.items():
    if 'SGD' in name:
        model.fit(X_train_sc, y_train); yp = model.predict(X_test_sc); yprob = model.decision_function(X_test_sc)
    else:
        model.fit(X_train, y_train); yp = model.predict(X_test); yprob = model.predict_proba(X_test)[:, 1]
    results[name] = {'pred': yp, 'prob': yprob, 'auc': roc_auc_score(y_test, yprob)}
    print(f"\n{'='*50}\n{name} | AUC: {results[name]['auc']:.4f}\n{'='*50}")
    print(classification_report(y_test, yp, target_names=['Active', 'Churned']))

## Step 4: ROC Analysis & Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['prob'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", linewidth=2)
axes[0].plot([0,1],[0,1],'k--',alpha=0.3)
axes[0].set_title('ROC Curves', fontweight='bold'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

rf_imp = pd.Series(results['Random Forest']['pred'], index=feature_cols) if False else \
    pd.Series(models['Random Forest'].feature_importances_, index=feature_cols).sort_values()
rf_imp.plot(kind='barh', ax=axes[1], color='#2563EB')
axes[1].set_title('Feature Importance (RF)', fontweight='bold'); axes[1].grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

## Step 5: Threshold Optimization & Revenue Impact

In [ ]:
rf_prob = results['Random Forest']['prob']
thresholds = np.arange(0.1, 0.9, 0.01)
recall_vals, precision_vals = [], []
for thresh in thresholds:
    yp = (rf_prob >= thresh).astype(int)
    cm = confusion_matrix(y_test, yp); tn, fp, fn, tp = cm.ravel()
    recall_vals.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    precision_vals.append(tp / (tp + fp) if (tp + fp) > 0 else 0)

target_recall = 0.80
valid = [(t, r, p) for t, r, p in zip(thresholds, recall_vals, precision_vals) if r >= target_recall]
optimal_threshold = max(valid, key=lambda x: x[2])[0] if valid else thresholds[np.argmax(recall_vals)]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, recall_vals, 'r-', label='Recall', linewidth=2)
ax.plot(thresholds, precision_vals, 'b-', label='Precision', linewidth=2)
ax.axvline(x=optimal_threshold, color='gray', linestyle='--', label=f'Optimal: {optimal_threshold:.2f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Precision-Recall Tradeoff by Threshold', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

y_pred_opt = (rf_prob >= optimal_threshold).astype(int)
cm = confusion_matrix(y_test, y_pred_opt); tn, fp, fn, tp = cm.ravel()
print(f"Optimal Threshold: {optimal_threshold:.2f}")
print(f"Recall: {tp/(tp+fn):.2%} | Precision: {tp/(tp+fp):.2%}")

## Step 6: Strategic Analysis — Churn Prevention Revenue Impact

In [ ]:
ARR = 100_000_000; avg_contract = 120_000
total_customers = ARR / avg_contract
churn_rate = y_test.mean(); recall = tp / (tp + fn)

print("=" * 70)
print("EXECUTIVE DASHBOARD: SaaS Churn Prevention ROI")
print("=" * 70)
print(f"\n1. PORTFOLIO: ${ARR:,.0f} ARR | {total_customers:,.0f} customers | {churn_rate:.1%} churn")
print(f"   Revenue at risk: ${ARR * churn_rate:,.0f}")

intervention_cost = 5000; intervention_success = 0.30
saved = tp * intervention_success
revenue_saved = saved * avg_contract
campaign_cost = (tp + fp) * intervention_cost
print(f"\n2. CAMPAIGN: {tp+fp} flagged | {tp} true churners | {saved:.0f} saved")
print(f"   Revenue saved: ${revenue_saved:,.0f} | Campaign cost: ${campaign_cost:,}")
print(f"   ROI: {(revenue_saved / campaign_cost - 1) * 100:.0f}%")

print(f"\n3. INTERVENTION PLAYBOOK:")
print(f"   🔴 High Risk (>0.7): Executive call + custom discount")
print(f"   🟡 Medium (0.4-0.7): CS check-in + training session")
print(f"   🟢 Low (<0.4): Automated health email + newsletter")